# Google Cloud workflow: BigQuery → Vertex AI → prediction endpoint
This notebook explains each cloud stage. **No cloud resources are created when it is run as delivered.** Start with `GCP_SETUP.md`, using a dedicated project with billing/trial credits linked. The project completed BigQuery preparation, Vertex training, endpoint prediction, and undeployment on October 14, 2026. Saved completion evidence and its provenance are in `artifacts/vertex/`.

In [1]:
import os
from pathlib import Path
import sys
if sys.version_info < (3, 11):
    raise RuntimeError("Select the Readmission GCP (Python 3.12) notebook kernel, restart it, and run all cells.")
from readmission.cloud import feature_sql
PROJECT_ID = "replace-with-your-project-id"
REGION = "us-central1"
DATASET = "readmission_demo"
BUCKET = f"{PROJECT_ID}-readmission-demo"
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/readmission-demo/readmission:v1"
print("Replace PROJECT_ID before executing cloud stages.")

Replace PROJECT_ID before executing cloud stages.


## 1. Review the BigQuery feature SQL
Raw columns are loaded as strings so missing markers and ICD-9 codes are preserved. Numeric features are explicitly cast. The same SHA256 patient bucket used locally is calculated in GoogleSQL. The ingest stage compares every prepared row and column against local preparation before Vertex training.

In [2]:
print(feature_sql('readmission-demo-example', DATASET))

-- Prediction time: discharge. Outcome and identifiers are excluded by the trainer.
-- 1: home; 6: home health service; 8: Home IV provider (UCI IDS_mapping.csv).
CREATE OR REPLACE TABLE `readmission-demo-example.readmission_demo.features`
OPTIONS(description='Diabetes-associated home discharges, patient-group split, recorded 30-day readmission target') AS
SELECT
  `encounter_id`,
  `patient_nbr`,
  COALESCE(NULLIF(NULLIF(`race`, '?'), ''), '__MISSING__') AS `race`,
  COALESCE(NULLIF(NULLIF(`gender`, '?'), ''), '__MISSING__') AS `gender`,
  CAST(`time_in_hospital` AS INT64) AS `time_in_hospital`,
  CAST(`num_lab_procedures` AS INT64) AS `num_lab_procedures`,
  CAST(`num_procedures` AS INT64) AS `num_procedures`,
  CAST(`num_medications` AS INT64) AS `num_medications`,
  CAST(`number_outpatient` AS INT64) AS `number_outpatient`,
  CAST(`number_emergency` AS INT64) AS `number_emergency`,
  CAST(`number_inpatient` AS INT64) AS `number_inpatient`,
  CAST(`number_diagnoses` AS INT64) AS `nu

## 2. Run the stages in Cloud Shell
Follow `GCP_SETUP.md` for exact commands. Separate stages make the work inspectable:

- `ingest`: load raw data, execute feature SQL, verify parity, export features to Cloud Storage.
- `train`: submit the custom Vertex job that executes `readmission.train`, including five-fold group CV and bagging.
- `collect`: confirm the job succeeded, then download measured metrics and the saved model.
- `deploy`: register the cloud-trained model and deploy its custom prediction container.
- `predict`: make an actual authenticated endpoint request and save the returned result.
- `cleanup`: undeploy serving compute after the demo.

The training job has a one-hour timeout and one CPU worker. Deployment uses one replica. Budget alerts notify you; they are not a hard spending cap.

In [3]:
commands = {
    "ingest": f"python -m readmission.cloud ingest --project {PROJECT_ID}",
    "train": f"python -m readmission.cloud train --project {PROJECT_ID} --image {IMAGE} --service-account readmission-train@{PROJECT_ID}.iam.gserviceaccount.com",
    "collect": f"python -m readmission.cloud collect --project {PROJECT_ID}",
    "deploy": f"python -m readmission.cloud deploy --project {PROJECT_ID} --image {IMAGE} --service-account readmission-serve@{PROJECT_ID}.iam.gserviceaccount.com",
    "predict": f"python -m readmission.cloud predict --project {PROJECT_ID}",
    "cleanup": f"python -m readmission.cloud cleanup --project {PROJECT_ID}",
}
for stage, command in commands.items():
    print(f"{stage}:\n  {command}\n")

ingest:
  python -m readmission.cloud ingest --project replace-with-your-project-id

train:
  python -m readmission.cloud train --project replace-with-your-project-id --image us-central1-docker.pkg.dev/replace-with-your-project-id/readmission-demo/readmission:v1 --service-account readmission-train@replace-with-your-project-id.iam.gserviceaccount.com

collect:
  python -m readmission.cloud collect --project replace-with-your-project-id

deploy:
  python -m readmission.cloud deploy --project replace-with-your-project-id --image us-central1-docker.pkg.dev/replace-with-your-project-id/readmission-demo/readmission:v1 --service-account readmission-serve@replace-with-your-project-id.iam.gserviceaccount.com

predict:
  python -m readmission.cloud predict --project replace-with-your-project-id

cleanup:
  python -m readmission.cloud cleanup --project replace-with-your-project-id



## 3. Inspect cloud evidence when it exists
Successful BigQuery jobs, the Vertex training resource, and endpoint status are written to `cloud-state.json`. A verified prediction is saved in `artifacts/vertex/endpoint_prediction.json`. Credentials are not included in the project or upload package.

In [4]:
import json
state_path = Path("cloud-state.json")
if state_path.exists():
    print(json.dumps(json.loads(state_path.read_text()), indent=2))
else:
    print("Cloud execution has not started. Complete the dedicated GCP project setup first.")

Cloud execution has not started. Complete the dedicated GCP project setup first.
